# Step 8 — CHiPS attention overlays

Renders, per slide, three panels on the whole-slide image: plain H&E, the
SurvCLAM attention heatmap, and the HPC (tile-cluster) overlay — each titled with
the slide's **CHiPS** score. Adapted from the PanColon-CHiPS study's WSI overlay
figure (`04_wsi_overlay_figure.ipynb`).

Run this **after** `python pancolon_pipeline.py attention_map ...` in the
`pancolon_survclam` environment. It reads only pipeline outputs + the config.


## 1. Load config and locate outputs

In [ ]:
import os, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import yaml  # config is plain YAML

CONFIG = os.environ.get("PANCOLON_CONFIG", "../config/pipeline.yaml")
with open(CONFIG) as fh:
    cfg = yaml.safe_load(fh)

repo_root = os.path.dirname(os.path.abspath(CONFIG)) + "/.."
def _abs(p):
    p = os.path.expanduser(str(p))
    return p if os.path.isabs(p) else os.path.normpath(os.path.join(repo_root, p))

work = _abs(cfg["paths"]["work_dir"])
dataset = cfg.get("dataset_name", "cohort")
model_key = cfg.get("build_pt", {}).get("model_key", "HPL_PANCOLON_20x")
wsi_dir = _abs(cfg["paths"]["wsi_dir"])

attention_dir = os.path.join(work, "attention")
att_pt_dir = os.path.join(attention_dir, "attention")     # per-slide *.pt dicts
tile_ids_dir = os.path.join(work, "datasets", dataset, model_key, "tile_ids")
chips_csv = os.path.join(work, "survclam", "chips_scores.csv")
hpc_csv = os.path.join(work, "clusters", f"{dataset}_hpc_assignment.csv")

print("attention pt dir:", att_pt_dir)
print("tile ids dir    :", tile_ids_dir)
print("chips scores    :", chips_csv)


## 2. CHiPS scores + helpers

In [ ]:
chips = pd.read_csv(chips_csv) if os.path.isfile(chips_csv) else pd.DataFrame()
id_col = "case_id" if "case_id" in chips.columns else "slide_id"
chips_lookup = dict(zip(chips[id_col].astype(str), chips["chips_score"])) if len(chips) else {}
pct_lookup = dict(zip(chips[id_col].astype(str), chips["chips_percentile"])) if "chips_percentile" in chips.columns else {}

try:
    import openslide
except ImportError:
    openslide = None
    print("openslide not importable; H&E panels will be skipped.")


def load_attention(slide_id):
    """Return (tile_ids, attention[np.ndarray]) for a slide, or (None, None)."""
    import torch
    cand = [os.path.join(att_pt_dir, f"{slide_id}.pt"),
            os.path.join(att_pt_dir, f"{slide_id}_BAG.pt")]
    cand += glob.glob(os.path.join(att_pt_dir, f"*{slide_id}*.pt"))
    for p in cand:
        if os.path.isfile(p):
            d = torch.load(p, map_location="cpu")
            attn = d.get("attention")
            attn = attn.numpy().reshape(-1) if hasattr(attn, "numpy") else np.asarray(attn).reshape(-1)
            return [str(t) for t in d.get("tile_ids", [])], attn
    return None, None


def load_tile_coords(slide_id):
    """Map tile_id -> (x, y). Prefers x/y columns; else parses '<slide>_<x>_<y>'."""
    cand = glob.glob(os.path.join(tile_ids_dir, f"*{slide_id}*.csv"))
    if not cand:
        return {}
    df = pd.read_csv(cand[0])
    tcol = next((c for c in ["tile_id","tiles","tile"] if c in df.columns), df.columns[0])
    xcol = next((c for c in ["x","tile_x","coord_x","w"] if c in df.columns), None)
    ycol = next((c for c in ["y","tile_y","coord_y","h"] if c in df.columns), None)
    coords = {}
    for _, r in df.iterrows():
        tid = str(r[tcol])
        if xcol and ycol:
            coords[tid] = (float(r[xcol]), float(r[ycol]))
        else:
            parts = tid.replace(".jpeg","").replace(".jpg","").split("_")
            try:
                coords[tid] = (float(parts[-2]), float(parts[-1]))
            except (ValueError, IndexError):
                pass
    return coords


## 3. Render overlays for selected slides

By default this shows the highest- and lowest-CHiPS slides. Edit `slides` to
pick your own. If a WSI file isn't found the H&E panel is left blank but the
attention/HPC scatter panels still render in tile-coordinate space.


In [ ]:
def find_wsi(slide_id):
    for ext in ("svs","ndpi","tif","tiff","mrxs"):
        hits = glob.glob(os.path.join(wsi_dir, f"*{slide_id}*.{ext}"))
        if hits:
            return hits[0]
    return None


def overlay_slide(slide_id, ax_row):
    chips_val = chips_lookup.get(slide_id)
    pct = pct_lookup.get(slide_id)
    title_suffix = ""
    if chips_val is not None:
        title_suffix = f"  |  CHiPS={chips_val:.3f}"
        if pct is not None:
            title_suffix += f" (p{pct:.0f})"

    tile_ids, attn = load_attention(slide_id)
    coords = load_tile_coords(slide_id)

    # Panel 1: H&E thumbnail
    ax = ax_row[0]
    wsi = find_wsi(slide_id)
    if openslide and wsi:
        slide = openslide.OpenSlide(wsi)
        thumb = slide.get_thumbnail((1024, 1024))
        ax.imshow(thumb)
    ax.set_title(f"{slide_id} — H&E{title_suffix}")
    ax.axis("off")

    # Panels 2-3: attention + HPC scatter in tile space
    if tile_ids is not None and coords:
        xy = np.array([coords.get(t, (np.nan, np.nan)) for t in tile_ids], float)
        ok = ~np.isnan(xy).any(axis=1)
        ax2 = ax_row[1]
        sc = ax2.scatter(xy[ok,0], -xy[ok,1], c=attn[ok] if attn is not None else None,
                         cmap="inferno", s=6)
        ax2.set_title("Attention"); ax2.set_aspect("equal"); ax2.axis("off")
        plt.colorbar(sc, ax=ax2, fraction=0.046)

        ax3 = ax_row[2]
        if os.path.isfile(hpc_csv):
            hpc = pd.read_csv(hpc_csv)
            hpc = hpc[hpc["slide_id"].astype(str).str.contains(slide_id, regex=False)]
            m = {str(r["tile_id"]): r["hpc"] for _, r in hpc.iterrows()} if "tile_id" in hpc.columns else {}
            labels = np.array([m.get(t, -1) for t in tile_ids])
            ax3.scatter(xy[ok,0], -xy[ok,1], c=labels[ok], cmap="tab20", s=6)
        ax3.set_title("HPC"); ax3.set_aspect("equal"); ax3.axis("off")
    else:
        for ax in ax_row[1:]:
            ax.set_title("(no attention/coords found)"); ax.axis("off")


# choose slides: highest and lowest CHiPS
if len(chips):
    ordered = chips.sort_values("chips_score", ascending=False)[id_col].astype(str).tolist()
    slides = ordered[:1] + ordered[-1:]
else:
    slides = [os.path.basename(p).split(".")[0]
              for p in glob.glob(os.path.join(att_pt_dir, "*.pt"))[:2]]

fig, axes = plt.subplots(len(slides), 3, figsize=(15, 5*len(slides)))
if len(slides) == 1:
    axes = axes[None, :]
for i, sid in enumerate(slides):
    overlay_slide(sid, axes[i])
plt.tight_layout()
plt.savefig(os.path.join(attention_dir, "chips_overlays.png"),
            dpi=cfg.get("attention", {}).get("overlay_dpi", 150), bbox_inches="tight")
plt.show()
print("saved ->", os.path.join(attention_dir, "chips_overlays.png"))
